In [1]:
import pandas as pd
import time
import os
import google.generativeai as genai
from tqdm import tqdm
# from google.colab import drive

In [ ]:
# --- CONFIGURATION ---
API_KEY = ""
MODEL_ID = "gemini-2.5-pro"
START_ROW = 0
END_ROW = 50

# File paths
INPUT_FILENAME = 'test.csv'
OUTPUT_FILENAME = 'test_0_49.csv'

genai.configure(api_key=API_KEY)

def translate_text(text):
    """
    Tries to translate text. 
    - Retries automatically on 503 (Overloaded) and 429 (Rate Limit).
    """
    max_retries = 5
    base_delay = 5

    for attempt in range(max_retries):
        try:
            model = genai.GenerativeModel(MODEL_ID)
            response = model.generate_content(
                f"Translate to English. Output ONLY the translation. Text: {text}"
            )
            return response.text.strip()

        except Exception as e:
            error_str = str(e)
            
            if "503" in error_str or "overloaded" in error_str:
                wait_time = base_delay * (2 ** attempt)
                print(f"\nServer overloaded (503). Retrying in {wait_time}s...")
                time.sleep(wait_time)
                continue
            
            elif "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
                wait_time = 4  # Wait 4 seconds for rate limit
                print(f"\nRate limit hit (429). Waiting {wait_time}s before retry...")
                time.sleep(wait_time)
                continue
            
            else:
                print(f"\nUnexpected Error: {e}")
                return None

    return None

def main():
    # Ensure input file exists
    if not os.path.exists(INPUT_FILENAME):
        print(f"Error: Could not find input file: {INPUT_FILENAME}")
        return

    # Resume Logic
    if os.path.exists(OUTPUT_FILENAME):
        print(f"Resuming from existing file: {OUTPUT_FILENAME}")
        df = pd.read_csv(OUTPUT_FILENAME)
    else:
        print(f"Starting fresh. Reading: {INPUT_FILENAME}")
        df = pd.read_csv(INPUT_FILENAME)
        if 'better_english_translation' not in df.columns:
            df['better_english_translation'] = None

    df['better_english_translation'] = df['better_english_translation'].astype(object)
    
    missing_rows = df[df['better_english_translation'].isnull()].index
    
    rows_to_process = [
        idx for idx in missing_rows 
        if idx >= START_ROW and (END_ROW is None or idx < END_ROW)
    ]
    
    print(f"Total rows in file: {len(df)}")
    print(f"Rows missing translation: {len(missing_rows)}")
    print(f"Configuration: START_ROW={START_ROW}, END_ROW={END_ROW}")
    print(f"Actual rows to process: {len(rows_to_process)}")
    print(f"Saving every rows...")
    print("------------------------------------------------")

    count = 0
    
    for idx in tqdm(rows_to_process):
        original_text = df.at[idx, 'text']
        
        if pd.isna(original_text) or str(original_text).strip() == "":
            df.at[idx, 'better_english_translation'] = ""
            continue

        translation = translate_text(original_text)
        
        if translation:
            df.at[idx, 'better_english_translation'] = translation
        
        count += 1

        # save every 1 translation
        if count % 1 == 0:
            df.to_csv(OUTPUT_FILENAME, index=False)
        
        time.sleep(30)

    # Final Save
    df.to_csv(OUTPUT_FILENAME, index=False)
    print(f"✅ Process ended. Saved final file to: {OUTPUT_FILENAME}")

if __name__ == "__main__":
    main()

Resuming from existing file: test_0_49.csv
Total rows in file: 200
Rows missing translation: 150
Configuration: START_ROW=0, END_ROW=50
Actual rows to process: 0
Saving every rows...
------------------------------------------------


0it [00:00, ?it/s]

✅ Process ended. Saved final file to: test_0_49.csv
